# News pulse — when does the stream breathe?

The `ngrams` table only stores *when* each phrase was seen, but that is
enough to reconstruct the rhythm of the news cycle: daily volume, and the
classic weekday × hour heatmap that reveals working hours, weekends and
quiet nights.

*SQL is kept inline in this notebook.*

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import psycopg
from dotenv import load_dotenv

# Connection settings come from the repository root .env
load_dotenv(Path("..") / ".env")

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DATABASE"],
    sslmode=os.getenv("PGSSLMODE", "require"),
)


def q(sql: str, **params) -> pd.DataFrame:
    """Run a query and return a DataFrame. Params are passed safely to psycopg."""
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


print("Ready — connected settings for", os.environ["POSTGRES_HOST"])


## Overall volume over time
How many n-grams land per hour.

In [ ]:
hourly = q("""
    SELECT date_trunc('hour', timestamp) AS hour,
           COUNT(*)                       AS n
    FROM ngrams
    GROUP BY 1
    ORDER BY 1
""")

fig = px.area(hourly, x="hour", y="n", title="n-grams ingested per hour")
fig.update_layout(height=400, xaxis_title="", yaxis_title="occurrences")
fig.show()

## The weekly heartbeat
Average activity by weekday and hour of day (UTC). Bright cells are the
busiest slots — usually weekday business hours.

In [ ]:
grid = q("""
    SELECT EXTRACT(DOW  FROM timestamp)::int AS dow,
           EXTRACT(HOUR FROM timestamp)::int AS hour,
           COUNT(*)                          AS n
    FROM ngrams
    GROUP BY 1, 2
""")

days = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
grid["day"] = grid["dow"].map(dict(enumerate(days)))

pivot = (
    grid.pivot_table(index="day", columns="hour", values="n", fill_value=0)
    .reindex(days)
)

fig = px.imshow(
    pivot,
    color_continuous_scale="Magma",
    aspect="auto",
    labels=dict(color="occurrences", x="hour of day", y=""),
    title="Weekday × hour activity heatmap",
)
fig.update_layout(height=420)
fig.show()